In [9]:
# -*- coding: utf-8 -*-
# 【CP1-08 MessagesState+LLM】对话状态基类与 LLM 联动的标准模板
# 文件：CP1/08_message_llm.ipynb
# 作用：本 Cell 演示 【CP1-08 MessagesState+LLM】对话状态基类与 LLM 联动的标准模板 的完整可运行示例
# 阅读顺序：状态定义 → 节点定义 → 图构建 → 编译执行 → 结果观察

from pyexpat import model

from langgraph.graph import StateGraph,START,END
from langchain_core.messages import HumanMessage
from langgraph.graph.message import MessagesState
from typing import TypedDict,Annotated
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
load_dotenv()

model = ChatDeepSeek(
    model = "deepseek-v4-flash",
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    }
)
class OverAllState(MessagesState):
    username:str
    output:str

def node_a(state:OverAllState) -> OverAllState:
    return{
        "messages" : [HumanMessage("你好，我是"+state["username"])],
    }

def llm_node(state : OverAllState) -> OverAllState:
    res = model.invoke(state["messages"])
    return {
        "messages" : [res],
        "output" : res.content
    }

builder = StateGraph(state_schema=OverAllState)  # 创建状态图构建器：绑定状态 Schema
builder.add_node(node_a)  # 注册节点到图中
builder.add_node(llm_node)  # 注册节点到图中
builder.add_edge(START,"node_a")  # 起点扇出
builder.add_edge("node_a","llm_node")
builder.add_edge("llm_node",END)  # 汇入终点

graph=builder.compile()  # 编译图：蓝图→可执行对象

result = graph.invoke({"username":"张三"})  # 触发图执行：传入初始 State + config
print(result)


{'messages': [HumanMessage(content='你好，我是张三', additional_kwargs={}, response_metadata={}, id='19c64e05-208a-4f7b-bfa6-1ddef30f3e6f'), AIMessage(content='你好，张三！很高兴认识你。我是DeepSeek，一个由深度求索公司创造的AI助手。有什么我可以帮你的吗？无论是回答问题、提供建议，还是陪你聊聊天，我都很乐意！😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 8, 'total_tokens': 54, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '138e098e-7764-4740-9c10-00202eba74d4', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f9520-cdfa-77e1-a9de-adc7d05115b9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 46, 'total_tokens': 54, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})